# Podcast-to-Blog Summarizer

Turn audio podcasts into structured blog posts with one click.

## Pipeline
1. **Transcribe** audio (MP3, WAV, M4A) using OpenAI Whisper
2. **Transform** transcript into a blog post using a local LLM (Llama)
3. **Output** includes: intro, main points, memorable quotes, and key takeaway

## Requirements
- **Colab**: Add `OPENAI_API_KEY` and `HF_TOKEN` to Notebook Secrets
- **Local**: Create `.env` with `OPENAI_API_KEY` and `HF_TOKEN`
- GPU recommended (T4 or better) for the LLM

## Tech Stack
`OpenAI Whisper · HuggingFace Transformers · BitsAndBytes · Gradio`

## Install dependencies

In [ ]:
# bitsandbytes>=0.46.1 required for 4-bit quantization
!pip install -q torch bitsandbytes>=0.46.1 transformers accelerate sentencepiece openai python-dotenv gradio

## Imports and configuration

In [ ]:
import os
from pathlib import Path

from openai import OpenAI
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch
import gradio as gr

# Constants
AUDIO_MODEL = "whisper-1"
LLM_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Supported audio extensions
AUDIO_EXTENSIONS = {".mp3", ".wav", ".m4a", ".webm", ".ogg", ".flac"}

## Load API keys (Colab or local)

In [ ]:

def get_api_keys():
    try:
        from google.colab import userdata
        openai_key = userdata.get("OPENAI_API_KEY")
        hf_token = userdata.get("HF_TOKEN")
        is_colab = True
    except Exception:
        from dotenv import load_dotenv
        load_dotenv(override=True)
        openai_key = os.getenv("OPENAI_API_KEY")
        hf_token = os.getenv("HF_TOKEN")
        is_colab = False
    return openai_key, hf_token, is_colab

openai_key, hf_token, _ = get_api_keys()

if not openai_key:
    raise ValueError("OPENAI_API_KEY not found. Add to Colab Secrets or .env file.")
if not hf_token:
    raise ValueError("HF_TOKEN not found. Add to Colab Secrets or .env file.")

openai_client = OpenAI(api_key=openai_key)

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=True)

print("API keys loaded successfully.")

## Load LLM and tokenizer

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    device_map="auto",
    quantization_config=quant_config,
)

print(f"Model loaded: {LLM_MODEL}")

## Transcribe and generate blog

In [ ]:
BLOG_SYSTEM_PROMPT = """You are an assistant that turns podcast transcripts into engaging blog posts.

Write in Markdown with these sections:
1. **Introduction** (2-3 sentences) - Hook the reader and summarize the topic
2. **Main Points** - Bullet points or numbered list of key ideas discussed
3. **Memorable Quotes** - 2-4 standout quotes from the transcript (use blockquotes)
4. **Key Takeaway** - One paragraph wrapping up the main lesson or insight

Use clear headings, keep it scannable, and preserve the original tone."""


def transcribe_audio(audio_path: str) -> str:
    """Transcribe audio file using OpenAI Whisper."""
    with open(audio_path, "rb") as f:
        transcript = openai_client.audio.transcriptions.create(
            model=AUDIO_MODEL,
            file=f,
            response_format="text",
        )
    return transcript


def transcript_to_blog(transcript: str) -> str:
    """Convert transcript to blog post using the LLM."""
    messages = [
        {"role": "system", "content": BLOG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Turn this podcast transcript into a blog post:\n\n{transcript}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to("cuda")
    outputs = model.generate(inputs, max_new_tokens=1500)
    response = tokenizer.decode(outputs[0])
    # Keep only the assistant reply (after chat template markers)
    if "<|end_header_id|>" in response:
        response = response.split("<|end_header_id|>")[-1].strip()
    response = response.replace("<|eot_id|>", "").strip()
    return response


def process_podcast(audio_file, progress=gr.Progress()):
    """Full pipeline: audio -> transcript -> blog."""
    if audio_file is None:
        return "Please upload an audio file.", ""

    # Handle Gradio Audio: can be (path, sample_rate) or path string
    path = audio_file[0] if isinstance(audio_file, tuple) else (audio_file if isinstance(audio_file, str) else getattr(audio_file, "name", str(audio_file)))
    ext = Path(path).suffix.lower()
    if ext not in AUDIO_EXTENSIONS:
        return f"Unsupported format. Use: {', '.join(AUDIO_EXTENSIONS)}", ""

    try:
        progress(0.2, desc="Transcribing audio...")
        transcript = transcribe_audio(path)

        progress(0.6, desc="Generating blog post...")
        blog = transcript_to_blog(transcript)

        progress(1.0, desc="Done!")
        return blog, transcript
    except Exception as e:
        return f"Error: {str(e)}", ""

## Gradio interface

In [ ]:
demo = gr.Interface(
    fn=process_podcast,
    inputs=gr.Audio(type="filepath", label="Upload podcast audio", sources=["upload"]),
    outputs=[
        gr.Markdown(label="Blog post", min_height=400),
        gr.Textbox(label="Transcript", lines=8, max_lines=20),
    ],
    title="Podcast-to-Blog Summarizer",
    description="Upload a podcast (MP3, WAV, M4A) to get a blog post with intro, main points, quotes, and takeaway. First run may take a few minutes to load the model.",
    flagging_mode="never",
)

demo.launch()